# ArabicaAI · Contact center & fraud prevention
## tl;dr
**Reviewed full-data snapshot · 26 September 2026.** 4,389 CSV files verified by S3 object name and size. Event dates span 17 June 2023–18 June 2026; process partitions end 17 June 2026. These are synthetic data, not real-bank rates.

- **Contact demand:** 240,056 / 686,296 contacts (34.98%) are `Transaccional`, with documented FCR 91.51%. This broad label does not identify specific transaction disputes.
- **Service friction:** `Queja` has 117,021 contacts, documented FCR 43.60% (51,021/117,021), and follow-up 62.97% (73,686/117,021). It accounts for 66,000/160,266 recorded unresolved contacts (41.18%). These three flags have no unknown values in this run.
- **Fraud:** 4,316/4,425,008 transactions are labeled fraud (0.0975%); unknown labels: 0. Channel and amount-band differences are exploratory, not validated prevention rules.
- **Linkage limit:** all 67,095 complaints lack `origin_interaction_id`; there is no documented complaint-to-transaction key. Dispute intake remains a conditional candidate, not a proven end-to-end historical workflow.

**Recommendation for discussion:** prioritize a narrow complaint/suspicious-charge intake and human-handoff slice, with transactional inquiries as the lower-risk comparator. Continue fraud-prevention feature and label-timing investigation separately. No prevention model or business improvement has been demonstrated.

The narrative describes this reviewed snapshot. If inputs change, update it after reviewing the new outputs; the manifest hash and execution time are recorded below.

## Context & Methods
Official sources: `Docs/sources/LATAM_Bank_Complete_Data_Dictionary (1).pdf` pp. 7–12; dataset summary pp. 1–5; hackathon problem statement pp. 2–6; `BUSINESS_OUTCOMES.md`. The dictionary is the schema reference; actual CSV values are reported without translating categories.

### Key Assumptions
- Synthetic historical records do not establish real-bank prevalence or causal effects. All verified supplied partitions are used. Source timestamps have no documented timezone; no timezone is inferred.
- Primary metrics retain only nonblank IDs occurring exactly once across all partitions. Every duplicate occurrence is excluded; raw-row comparisons reveal sensitivity. No silent deduplication.
- Rates use known valid values and display numerator, denominator and unknown count. Documented FCR is the supplied `was_resolved` flag, not independently proven absence of repeat contacts. Unknown is never false.
- Customer country enrichment uses only unambiguous dimension keys; it is a current snapshot, not historical country at event time. Facts are never joined directly to each other by customer ID.
- Open complaints are excluded from resolved-case duration summaries. Money is summed within currency; USD missingness remains visible. Fraud-labeled amounts are exposure, not realized loss.
- `is_fraud` is the target; `fraud_score` needs timing/provenance verification; `transaction_status` and `response_code` may be post-decision. No model is trained. Any future history feature must use only prior events.
- Exact duplicate counts and sorts are disk-backed. Python retains bounded batches and compact aggregates; SQLite uses a 32 MiB cache. Input verification compares object names/sizes, not cryptographic S3 content checksums.

### 1. Parameters and environment
Run from the repository or its `notebooks/` directory. Override paths through the environment variables below. See `notebooks/README.md` for setup and manifest generation. A full run rebuilds an ignored analytical SQLite file and may take several minutes.

In [ ]:
import os, sys, json, sqlite3, hashlib, platform
from pathlib import Path
from datetime import datetime, timezone
from contextlib import closing
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, StrMethodFormatter
from IPython.display import display, Markdown

PROJECT_ROOT = Path(os.environ.get('ARABICA_REPO', Path.cwd())).resolve()
if not (PROJECT_ROOT / 'data_foundation').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'data_foundation').is_dir(), 'Set ARABICA_REPO to the repository root'
sys.path.insert(0, str(PROJECT_ROOT))
from data_foundation.src.exploration import FIELDS, verify_inputs, load_table, summarize

DATA_ROOT = Path(os.environ.get('ARABICA_DATA_ROOT', PROJECT_ROOT / 'data')).resolve()
SCRATCH_ROOT = Path(os.environ.get('ARABICA_SCRATCH_ROOT', PROJECT_ROOT / 'data_foundation/runs/meeting-exploration')).resolve()
MANIFEST_PATH = Path(os.environ.get('ARABICA_MANIFEST_PATH', SCRATCH_ROOT / 'manifest.json'))
RUN_LABEL = os.environ.get('ARABICA_RUN_LABEL', 'full')
BATCH_SIZE = 5000
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
assert SCRATCH_ROOT != DATA_ROOT and not SCRATCH_ROOT.is_relative_to(DATA_ROOT), 'Keep scratch outside raw data'
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.axisbelow': True, 'figure.dpi': 120})
BLUE, ORANGE, GRAY = '#2463A5', '#B85C22', '#62666D'
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 12)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
print('Run:', RUN_LABEL, '| Python:', platform.python_version())

def show(frame):
    """Render bounded tables with legible rates and explicit unknowns."""
    if frame.index.name is not None:
        frame = frame.reset_index()
    formatters = {name: '{:.3%}' for name in ['rate','eligible_share','raw_share','usd_coverage'] if name in frame.columns}
    display(frame.style.format(precision=2, thousands=',', na_rep='Unknown').format(formatters, na_rep='Unknown').hide(axis='index'))


## Data
### 2. Verify inputs, load once, calculate
The manifest lists exact expected CSV objects. Missing/extra files, size mismatches, malformed rows and required-column failures stop execution. Every file is checked for changes while it is read. The output below identifies raw and analytical populations; published tables contain aggregates only.

In [ ]:
manifest_bytes = MANIFEST_PATH.read_bytes()
manifest = json.loads(manifest_bytes)
inputs = verify_inputs(DATA_ROOT, manifest, tuple(FIELDS))
input_stats = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for paths in inputs.values() for p in paths}
with closing(sqlite3.connect(SCRATCH_ROOT / 'meeting.sqlite')) as connection:
    quality = [load_table(connection, table, paths, BATCH_SIZE) for table, paths in inputs.items()]
    results = summarize(connection)
assert all((Path(p).stat().st_size, Path(p).stat().st_mtime_ns) == stats for p, stats in input_stats.items()), 'Input changed during run'
verify_inputs(DATA_ROOT, manifest, tuple(FIELDS))
for row in results['rates']:
    assert row['numerator'] <= row['denominator'] <= row['eligible']
    assert row['unknown'] + row['denominator'] == row['eligible']
for row in quality:
    assert row['raw'] == row['eligible'] + row['blank_rows'] + row['duplicate_rows']
provenance = {'run_label': RUN_LABEL, 'executed_utc': datetime.now(timezone.utc).isoformat(),
              'manifest_sha256': hashlib.sha256(manifest_bytes).hexdigest(),
              'verified_files': sum(map(len, inputs.values())),
              'verified_bytes': sum(stat[0] for stat in input_stats.values()),
              'python': platform.python_version(),
              'helper_sha256': hashlib.sha256((PROJECT_ROOT / 'data_foundation/src/exploration.py').read_bytes()).hexdigest()}
(SCRATCH_ROOT / 'results.json').write_text(json.dumps({'provenance': provenance, 'quality': quality, 'results': results}, indent=2))
print(json.dumps(provenance, indent=2))
quality_frame = pd.DataFrame(quality).drop(columns='fields')
show(quality_frame)

### 3. Coverage, unknowns and join quality
Dates below describe eligible records. Later process dates can reflect delayed arrival; event dates after the process date are flagged separately. Their meaning requires confirmation of timezone and processing-day semantics. Documentation row counts are approximate and are not substituted for measured counts.

**Observed:** the measured primary populations are 686,296 interactions, 4,425,008 transactions and 67,095 complaints—below the dictionary's approximate 800,000 / 5,000,000 / 80,000 counts despite complete required-object coverage. No blank or duplicate IDs were found in these five loaded tables. This is not a certification of every field or all 13 source tables. Event dates after process dates are common; clarify processing-day/timezone semantics before operational freshness rules.

In [ ]:
show(pd.DataFrame(results['coverage']))
field_quality = pd.DataFrame([{'table': q['table'], 'field_issue': field, 'raw_rows': count}
                             for q in quality for field, count in q['fields'].items()])
show(field_quality.sort_values('raw_rows', ascending=False).head(20) if not field_quality.empty else field_quality)
show(pd.DataFrame(results['enrichment']))

## Results
### 4. What brings customers to the contact center?
Bars count eligible interactions, not unique customers. The table compares raw-row shares with eligible shares so exclusions remain visible. Up to 100 categories per dimension are retained; figures show at most ten. Exact total counts remain untruncated.

**Observed:** transactional contacts represent 34.98% of interactions. `contact_reason` contains broad categories, so identifying a specific customer intent requires a separate labeled-text review. With no duplicate/blank IDs here, raw and eligible rankings coincide.

In [ ]:
rates, volumes = pd.DataFrame(results['rates']), pd.DataFrame(results['volumes'])
durations = pd.DataFrame(results['durations'])
def select(frame, **filters):
    for field, value in filters.items():
        frame = frame.loc[frame[field].eq(value)]
    return frame.copy()
def bars(frame, column, title, xlabel, percent=False, source_table="call_center_interactions"):
    data = frame.dropna(subset=[column]).head(10).sort_values(column)
    fig, ax = plt.subplots(figsize=(10, max(3.5, len(data) * .43 + 1)))
    if data.empty:
        ax.text(.5,.5,'No known values',ha='center',va='center',transform=ax.transAxes)
    else:
        ax.barh(data['label'].astype(str), data[column], color=BLUE)
        if percent:
            ax.xaxis.set_major_formatter(PercentFormatter(1))
        else:
            ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    period = next(r for r in results['coverage'] if r['table'] == source_table)
    span = f"{str(period['first_event'])[:10]} to {str(period['last_event'])[:10]}"
    ax.set(title=title + '\n' + span, xlabel=xlabel, xlim=(0, None))
    ax.grid(axis='x',alpha=.18)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

contact = select(volumes, table='call_center_interactions', dimension='contact_reason')
sensitivity = contact.pivot(index='label',columns='population',values='rows').fillna(0)
sensitivity['eligible_share'] = sensitivity['eligible'] / max(1, next(q['eligible'] for q in quality if q['table']=='call_center_interactions'))
sensitivity['raw_share'] = sensitivity['raw'] / max(1, next(q['raw'] for q in quality if q['table']=='call_center_interactions'))
sensitivity['share_change_pp'] = 100*(sensitivity['eligible_share'] - sensitivity['raw_share'])
show(sensitivity.sort_values('eligible',ascending=False).head(15))
bars(select(contact,population='eligible'), 'rows', 'Contact reasons · supplied history', 'Eligible interactions')

### 5. Resolution, follow-up and escalation
Rates use known values for each flag; their denominators can differ. The chart is descriptive and does not establish that one type of contact causes escalation. The consistency table checks cases marked resolved that still require follow-up.

**Observed:** complaint-related contacts have 43.60% documented FCR versus 91.51% for transactional contacts. Their median measured duration is 431 seconds (100,727 known values) versus 205 seconds (206,465 known values). Duration is missing for 96,234/686,296 contacts; wait time for 205,618/686,296. Also, 78,788 contacts are marked both resolved and requiring follow-up: clarify the flag semantics before calling this independently verified first-contact resolution.

In [ ]:
contact_rates = select(rates,table='call_center_interactions',dimension='contact_reason')
show(contact_rates[['label','metric','numerator','denominator','unknown','eligible','rate']].head(40))
plot_data = contact_rates[contact_rates.metric.isin(['was_resolved','requires_followup','was_escalated'])].pivot(index='label',columns='metric',values='rate')
if not plot_data.empty:
    ax = plot_data.rename(columns={'was_resolved':'Documented FCR','requires_followup':'Follow-up','was_escalated':'Escalated'}).plot.barh(figsize=(11,6),color=[BLUE,ORANGE,GRAY])
    ax.set(xlabel='Share of known flag values',title='Contact outcomes by reason',xlim=(0,1))
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.legend(loc='lower right',fontsize=9)
    plt.tight_layout(); plt.show(); plt.close()
show(pd.DataFrame(results['consistency']))
show(select(durations,table='call_center_interactions')[['metric','dimension','label','known','median','p95']])

### 6. Channel, country and time
Country is a customer snapshot and is unknown for missing or ambiguous customer keys. Boundary months may be partial; a low count in a partial month is not a demand decline.

In [ ]:
show(select(rates,table='call_center_interactions',dimension='channel',metric='was_resolved')[['label','numerator','denominator','unknown','rate']])
show(select(rates,table='call_center_interactions',dimension='country',metric='was_resolved')[['label','numerator','denominator','unknown','rate']])
monthly = select(rates,table='call_center_interactions',dimension='month',metric='was_resolved')
monthly = monthly.loc[monthly.label.ne('Unknown')].sort_values('label')
if len(monthly) > 1:
    fig,ax = plt.subplots(figsize=(11,4))
    ax.plot(pd.to_datetime(monthly.label), monthly['rate'], marker='o',markersize=3,color=BLUE)
    ax.set(title='Documented FCR by event month · boundary months may be partial',ylabel='Resolved / known outcomes',ylim=(0,1))
    ticks = sorted(set(pd.to_datetime(monthly.label).iloc[::6].tolist() + [pd.to_datetime(monthly.label).iloc[-1]]))
    ax.set_xticks(ticks); ax.set_xticklabels([t.strftime('%b %Y') for t in ticks])
    ax.yaxis.set_major_formatter(PercentFormatter(1)); fig.autofmt_xdate(rotation=0)
    fig.tight_layout(); plt.show(); plt.close(fig)

### 7. Supporting complaint and language evidence
Complaint records have their own grain; they are not added to interaction counts. SLA breach is a supplied flag. Resolved-case duration includes only Resolved/Closed cases with valid creation/resolution dates, nonnegative duration and resolution on/after creation. Open cases remain visible and are not assigned zero duration. Transcript metadata is not evidence of full population language coverage.

**Observed:** complaint SLA breaches are 13,495/67,095 (20.11%). `Cargo no reconocido` appears on 12,297 complaints, and `Cobro indebido` on 12,194; these are complaint subcategory labels, not confirmed fraud. All 67,095 originating-interaction links are blank, so no complaint-to-call performance claim is supported. The 171,321 transcripts are all labeled Spanish (`es`) and have unambiguous interaction references; they cover about a quarter of contacts, not the full population. Portuguese evaluation cases still need to be built.

In [ ]:
complaint_rates = select(rates,table='complaints',dimension='category',metric='sla_breached')
show(complaint_rates[['label','eligible','numerator','denominator','unknown','rate']])
bars(complaint_rates.sort_values('rate',ascending=False), 'rate', 'Complaint SLA breaches by category', 'Breached / known SLA flags', True, 'complaints')
show(select(volumes,table='complaints',dimension='status',population='eligible')[['label','rows']])
show(select(durations,table='complaints')[['dimension','label','known','median','p95']])
show(pd.DataFrame(results['languages']))

### 8. Fraud labels: where is the observed exposure?
The following rates describe synthetic labeled transactions. They do not measure prevention effectiveness, model precision/recall or money recovered. Status breakdowns are descriptive outcomes, not assumed pre-authorization features. Unknown fraud labels remain outside the known-label denominator.

Country labels are preserved as supplied; `Mexico` and `México` can both appear. Treat these as a naming-quality issue, not different countries. Resolve aliases before using geography in a prevention rule. Small groups can produce unstable rates.

**Observed:** prevalence is 0.0975%; a constant non-fraud prediction would have 99.9025% accuracy while detecting none of the labeled fraud. This arithmetic illustrates why accuracy alone is unsuitable; it is not an evaluated prevention model. Channel rates range from 0.0948% (POS) to 0.1079% (Branch). Do not treat this small descriptive spread as a validated blocking policy.

In [ ]:
fraud_total = select(rates,table='transactions',dimension='all',metric='is_fraud')
show(fraud_total[['numerator','denominator','unknown','eligible','rate']])
fraud_channel = select(rates,table='transactions',dimension='channel',metric='is_fraud')
show(fraud_channel[['label','numerator','denominator','unknown','rate']])
bars(fraud_channel.sort_values('rate',ascending=False),'rate','Fraud-labeled transactions by channel','Fraud / known labels',True,'transactions')
show(select(rates,table='transactions',dimension='transaction_type',metric='is_fraud')[['label','numerator','denominator','unknown','rate']])
fraud_country = select(rates,table='transactions',dimension='transaction_country',metric='is_fraud')
show(fraud_country[['label','numerator','denominator','unknown','rate']])
bars(fraud_country.sort_values('rate',ascending=False),'rate','Fraud labels by transaction country','Fraud / known labels',True,'transactions')
show(select(rates,table='transactions',dimension='transaction_category',metric='is_fraud')[['label','numerator','denominator','unknown','rate']])
show(select(rates,table='transactions',dimension='transaction_status',metric='is_fraud')[['label','numerator','denominator','unknown','rate']])

### Prevention signals to investigate next
Fixed USD bands and event-hour groups examine variables potentially available at decision time. Bands are descriptive choices, not tuned thresholds. The timezone of event timestamps is unspecified, so hours must not be interpreted as customer-local time. Group differences are exploratory; an out-of-time test and label-availability audit are still needed.

In [ ]:
bands = select(rates,table='transactions',dimension='usd_band',metric='is_fraud')
show(bands[['label','numerator','denominator','unknown','eligible','rate']])
bars(bands.sort_values('rate',ascending=False),'rate','Fraud labels by supplied USD amount band','Fraud / known labels',True,'transactions')
hours = select(rates,table='transactions',dimension='event_hour',metric='is_fraud').sort_values('label')
show(hours[['label','numerator','denominator','unknown','rate']])
monthly_fraud = select(rates,table='transactions',dimension='month',metric='is_fraud')
monthly_fraud = monthly_fraud.loc[monthly_fraud.label.ne('Unknown')].sort_values('label')
if len(monthly_fraud) > 1:
    fig,ax = plt.subplots(figsize=(11,4))
    ax.plot(pd.to_datetime(monthly_fraud.label),monthly_fraud['rate'],color=BLUE,marker='o',markersize=3)
    ax.set(title='Fraud labels by event month · boundary months may be partial',ylabel='Fraud / known labels',ylim=(0,None))
    ticks = sorted(set(pd.to_datetime(monthly_fraud.label).iloc[::6].tolist() + [pd.to_datetime(monthly_fraud.label).iloc[-1]]))
    ax.set_xticks(ticks); ax.set_xticklabels([t.strftime('%b %Y') for t in ticks])
    ax.yaxis.set_major_formatter(PercentFormatter(1)); fig.autofmt_xdate(rotation=0)
    fig.tight_layout(); plt.show(); plt.close(fig)

### 9. Monetary coverage and prevention variable roles
Amounts below remain separated by original currency. `amount_usd` is an optional supplied conversion; no exchange rate is inferred. Missing USD values prevent a complete exposure estimate. Score differences by label can reflect how the synthetic data was generated and are not a validated model baseline.

**Observed:** the stored `amount_usd` field is missing on 2,537,456/4,425,008 rows (57.34%). This is largely structural: every native-USD row has a blank `amount_usd`, although `amount` is already in USD. The tables deliberately preserve both source fields; future normalized USD features should explicitly use native-USD `amount`, and only then assess remaining conversion gaps. The amount-band chart currently uses the supplied `amount_usd` only, so its known bands underrepresent native-USD transactions. `fraud_score` is also missing on 885,157 transactions; its provenance and availability at authorization time are not established.

In [ ]:
money = pd.DataFrame(results['money'])
for source, target in [('amount_cents','local_amount'), ('amount_usd_cents','usd_amount'), ('absolute_usd_cents','absolute_usd_exposure')]:
    money[target] = money[source] / 100
money['usd_coverage'] = money.usd_known / money.eligible
show(money[['currency','is_fraud','eligible','amount_known','local_amount','usd_known','usd_coverage','usd_amount','absolute_usd_exposure']])
show(select(durations,table='transactions')[['metric','label','known','median','p95']])
roles = pd.DataFrame([
    ['is_fraud','Target label','Never use as a predictor'],
    ['amount, currency, type, channel, location','Candidate input','Verify availability at decision time'],
    ['transaction_date + customer_id','Historical features','Only prior events; use time-based validation'],
    ['fraud_score','Existing derived score','Provenance/timing unknown; no baseline claim'],
    ['transaction_status, response_code','Potential post-decision outcome','Exclude from assumed prevention inputs'],
    ['process_date','Arrival/partition metadata','Do not substitute for event or label availability time'],
], columns=['Variables','Role','Constraint'])
show(roles)

## Takeaways
### Workflow comparison · evidence versus judgment

| Candidate | Observed evidence | Feasibility and data limits | Suggested baseline and metric |
|---|---|---|---|
| Complaint / suspicious-charge intake with human handoff | 117,021 `Queja` contacts; 43.60% documented FCR; 62.97% follow-up. 12,297 complaints carry `Cargo no reconocido`. These are separate populations. | Strong service-friction signal. No originating-call links and no complaint-to-transaction key; an end-to-end demo needs explicit authenticated transaction selection and clearly labeled mock case tools. Never decide fraud or promise a refund from these counts. | Fixed intake checklist versus learned intent/slot extraction; measure complete safe intake / all in-scope cases, missed/unnecessary escalation and verified handoff quality. |
| Transaction/account inquiry assistant | 240,056 `Transaccional` contacts (34.98%); 91.51% documented FCR. | Largest broad contact group, but the label does not isolate account inquiries. Lower-risk comparison using authorized read-only facts; narrower intent labels still needed. | Deterministic FAQ/routing versus learned retrieval/intent handling; safe verified resolution / all in-scope cases. |
| Fraud-prevention triage | 4,316 fraud labels across 4,425,008 transactions (0.0975%). | Rare target; no label-availability timestamp. Score and outcome-field timing unverified. Amount normalization and country aliases need explicit handling. Current descriptive analysis does not justify automated blocking. | Propose an out-of-time simple rules baseline after timing audit; compare precision/recall and recall at a fixed review budget, with false positives and denominators. |

**Decision recommendation (judgment):** take complaint/suspicious-charge intake to the meeting as the leading *conditional* service slice, retain transactional inquiry as the comparator, and give fraud prevention a separate leakage-safe feasibility task. There is no measured ROI or causal benefit in this notebook.

**Proposed next ownership decisions:** Roberto owns this scorecard and the prevention variable/label-timing audit; Andrés reviews transcript intents/slots; Lucas defines the baseline, primary KPI and held-out cases; Manoella establishes reproducible data/tool access. Confirm these assignments with the team; nothing has been sent to them.

**Minimum next proof:** a small reviewed set of normal, ambiguous and human-handoff cases; Spanish and Portuguese coverage; deterministic customer/action permissions; baseline and proposed system on the same held-out workload. Track unsafe cases with denominators, missed/unnecessary handoffs, latency and cost. A fraud classifier or a dashboard is not required to prove the intake slice.

### Validation
Twelve regression tests passed. The notebook ran top-to-bottom in clean kernels on a synthetic fixture, a one-partition smoke subset, a seven-partition controlled subset, and all required data. All contact-reason raw counts and resolution flags were independently recounted from CSV; transaction counts and USD sums were independently checked on one complete partition. This sampled monetary check is not an all-row monetary reconciliation. Visual inspection and independent code review complete the handoff; no training, publication or external messages were performed.